# 지식 그래프 구축: NetworkX / Neo4j / Cypher / LLM 엔티티 추출

1. NetworkX로 4M(Man·Machine·Material·Method) 불량 관계 그래프 직접 구성 + 시각화
2. Neo4j 연결 후 동일 그래프를 Cypher로 적재
3. Cypher multi-hop 경로 탐색 쿼리 실습
4. Ollama LLM으로 보고서 텍스트에서 엔티티·관계 triple 자동 추출


## 0. 환경 설정 및 패키지 설치

In [49]:
!pip install -q networkx pyvis neo4j


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 130.1 MB/s eta 0:00:0000:01


In [50]:
!wget https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/reports_structured.json
!wget https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/reports.json

--2026-06-23 22:00:52--  https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/reports_structured.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 140040 (137K) [text/plain]
Saving to: ‘reports_structured.json’

reports_structured. 100%[===================>] 136.76K  --.-KB/s    in 0.002s  

2026-06-23 22:00:53 (63.7 MB/s) - ‘reports_structured.json’ saved [140040/140040]

--2026-06-23 22:00:53--  https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/reports.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 126861

In [51]:
import json
import networkx as nx
from pyvis.network import Network

with open("reports_structured.json", "r", encoding="utf-8") as f:
    reports = json.load(f)

print(f"총 {len(reports)}건 로드")


총 150건 로드


## 1. NetworkX로 4M 불량 관계 그래프 직접 구성

스키마: 설비종류 → 불량유형 → 4M_분류 → 담당파트




In [52]:
# 예시 그래프 만들기 (구조 이해용)
G_example = nx.DiGraph()

G_example.add_node("CNC 가공기 실린더블록용 A-3호기", type="설비")
G_example.add_node("치수불량", type="불량유형")
G_example.add_node("Machine", type="4M분류")
G_example.add_node("엔진가공1팀", type="담당파트")

G_example.add_edge("CNC 가공기 실린더블록용 A-3호기", "치수불량", relation="발생")
G_example.add_edge("치수불량", "Machine", relation="원인분류")
G_example.add_edge("Machine", "엔진가공1팀", relation="조치담당")

print("노드:", list(G_example.nodes(data=True)))
print("\n엣지:", list(G_example.edges(data=True)))


노드: [('CNC 가공기 실린더블록용 A-3호기', {'type': '설비'}), ('치수불량', {'type': '불량유형'}), ('Machine', {'type': '4M분류'}), ('엔진가공1팀', {'type': '담당파트'})]

엣지: [('CNC 가공기 실린더블록용 A-3호기', '치수불량', {'relation': '발생'}), ('치수불량', 'Machine', {'relation': '원인분류'}), ('Machine', '엔진가공1팀', {'relation': '조치담당'})]


In [53]:
print(reports[0].keys())
print(reports[0])

dict_keys(['report_id', '설비명', '설비종류', '불량유형', '발생일시', '증상_설명', '4M_분류', '추정원인', '조치내용', '재발방지대책', '담당파트', '심각도'])
{'report_id': 'RPT-001', '설비명': 'CNC 머신 A-1호기', '설비종류': 'CNC 머신', '불량유형': '치수불량', '발생일시': '2024-01-03 09:15', '증상_설명': '가공 완료된 알루미늄 브래킷의 기준 홀 직경이 도면 12.00mm 대비 12.18mm로 측정되었다. 동일 로트 20개 중 7개에서 +0.16~+0.18mm 치수오차가 반복 확인되었고, 스핀들 진동은 0.18mm 수준이었다.', '4M_분류': 'Machine', '추정원인': '스핀들 베어링 마모로 회전 흔들림이 증가하여 홀 가공 시 런아웃이 발생한 것으로 판단된다.', '조치내용': '스핀들 베어링 상태를 점검하고 베어링 교체 후 런아웃을 재측정하였다. 시험 가공품 5개를 확인하여 홀 직경이 12.00±0.02mm 범위에 들어오는지 검증하였다.', '재발방지대책': '스핀들 진동이 0.12mm를 초과하면 즉시 설비보전팀에 점검 요청하도록 기준을 등록한다.', '담당파트': '설비보전팀', '심각도': '상'}


In [54]:
# 2단계: reports.json 전체로 그래프 확장
G = nx.DiGraph()

for r in reports:
    설비 = r["설비명"]
    불량유형 = r["불량유형"]
    분류4M = r["4M_분류"]
    담당파트 = r["담당파트"]
    report_id = r["report_id"]

    G.add_node(설비, type="설비")
    G.add_node(불량유형, type="불량유형")
    G.add_node(분류4M, type="4M분류")
    G.add_node(담당파트, type="담당파트")

    G.add_edge(설비, 불량유형, relation="발생", report_id=report_id)
    G.add_edge(불량유형, 분류4M, relation="원인분류", report_id=report_id)
    G.add_edge(분류4M, 담당파트, relation="조치담당", report_id=report_id)

print(f"총 노드 수: {G.number_of_nodes()}")
print(f"총 엣지 수: {G.number_of_edges()}")


총 노드 수: 126
총 엣지 수: 165


In [55]:
# 노드 타입별 개수 확인
from collections import Counter

node_types = Counter(nx.get_node_attributes(G, "type").values())
print("노드 타입별 개수:", dict(node_types))


노드 타입별 개수: {'설비': 105, '불량유형': 6, '4M분류': 4, '담당파트': 11}


In [57]:
# pyvis로 인터랙티브 시각화
net = Network(notebook=True, cdn_resources="in_line", height="600px", width="100%", directed=True)

color_map = {"설비": "#4C72B0", "불량유형": "#DD8452", "4M분류": "#55A868", "담당파트": "#C44E52"}

for node, attrs in G.nodes(data=True):
    net.add_node(node, label=node, color=color_map.get(attrs.get("type"), "#999999"))

for u, v, attrs in G.edges(data=True):
    net.add_edge(u, v, title=attrs.get("relation", ""))

net.show("4m_defect_graph.html")
print("그래프 시각화 파일 생성: 4m_defect_graph.html")


4m_defect_graph.html
그래프 시각화 파일 생성: 4m_defect_graph.html


## 2. NetworkX 기본 연산: 경로 탐색

특정 설비에서 출발해 어떤 4M 분류·담당파트로 이어지는지 multi-hop 경로를 탐색


In [58]:
# 특정 설비에서 도달 가능한 모든 노드 탐색
target_equipment = reports[0]["설비명"]
print(f"기준 설비: {target_equipment}")

reachable = nx.descendants(G, target_equipment)
print(f"도달 가능한 노드 수: {len(reachable)}")
for node in reachable:
    print(" -", node, f"({G.nodes[node]['type']})")


기준 설비: CNC 머신 A-1호기
도달 가능한 노드 수: 15
 - 사출2팀 (담당파트)
 - 용접2팀 (담당파트)
 - 품질관리팀 (담당파트)
 - 용접1팀 (담당파트)
 - 사출1팀 (담당파트)
 - 치수불량 (불량유형)
 - CNC가공팀 (담당파트)
 - 포장팀 (담당파트)
 - Machine (4M분류)
 - Man (4M분류)
 - 프레스가공팀 (담당파트)
 - Method (4M분류)
 - 공정관리팀 (담당파트)
 - 물류운반팀 (담당파트)
 - 설비보전팀 (담당파트)


In [59]:
# 최단 경로 탐색: 설비 -> 4M분류
target_4m = "Machine"

try:
    path = nx.shortest_path(G, source=target_equipment, target=target_4m)
    print(f"경로: {' -> '.join(path)}")
except nx.NetworkXNoPath:
    print(f"{target_equipment}에서 {target_4m}로 가는 경로가 없습니다.")


경로: CNC 머신 A-1호기 -> 치수불량 -> Machine


In [60]:
# 특정 4M 분류로 이어지는 모든 설비 역추적 (predecessors 활용)
target_4m = "Machine"
related_defect_types = list(G.predecessors(target_4m))
print(f"'{target_4m}' 원인으로 분류된 불량유형: {related_defect_types}")

for dt in related_defect_types:
    equipments = list(G.predecessors(dt))
    print(f"\n  '{dt}' 불량을 일으킨 설비들: {equipments[:5]}{'...' if len(equipments)>5 else ''}")


'Machine' 원인으로 분류된 불량유형: ['치수불량', '표면결함-스크래치/크랙', '조립불량', '이물질혼입', '누설']

  '치수불량' 불량을 일으킨 설비들: ['CNC 머신 A-1호기', 'CNC 머신 A-3호기', '사출성형기 #2호기', 'CNC 머신 C-1호기', '사출성형기 #5호기']...

  '표면결함-스크래치/크랙' 불량을 일으킨 설비들: ['CNC 머신 A-2호기', '사출성형기 #1호기', 'CNC 머신 B-2호기', '사출성형기 #4호기', 'CNC 머신 D-2호기']...

  '조립불량' 불량을 일으킨 설비들: ['CNC 머신 B-1호기', '사출성형기 #3호기', 'CNC 머신 C-2호기', '사출성형기 #6호기', 'CNC 머신 E-1호기']...

  '이물질혼입' 불량을 일으킨 설비들: ['컨베이어 벨트 1라인', '컨베이어 벨트 4라인', '컨베이어 벨트 7라인', '컨베이어 벨트 10라인', '컨베이어 벨트 13라인']...

  '누설' 불량을 일으킨 설비들: ['컨베이어 벨트 2라인', '컨베이어 벨트 5라인', '컨베이어 벨트 8라인', '컨베이어 벨트 11라인', '컨베이어 벨트 14라인']...


- 표(테이블)였다면 "이 설비가 결국 어떤 4M 원인까지 이어지는가"를 알기 위해 여러 테이블을 JOIN해야 하지만, 그래프에서는 경로 탐색으로 해결
- `descendants`/`predecessors`로 정방향/역방향 탐색이 모두 가능


## 3. Neo4j 연결 및 동일 그래프 적재

Neo4j Aura(무료 클라우드 인스턴스) 또는 로컬 Neo4j Desktop 연결 정보가 필요
1. https://neo4j.com/cloud/aura-free 접속 후 무료 계정 생성  
2. "Create Instance" → Free tier 선택  
3. 인스턴스 생성 시 Connection URI와 초기 비밀번호가 한 번만 표시됩니다(꼭 저장해두세요)  
4. 발급받은 값을 코드에 그대로 넣기:  cNULIOfPxWEBZ1IXFZSUrPcDivYlSrfB1zfGhlz2-zA

In [61]:
from neo4j import GraphDatabase

URI = "neo4j+s://ce6843cb.databases.neo4j.io"  # Aura면 neo4j+s
USER = "ce6843cb"
PASSWORD =  "cNULIOfPxWEBZ1IXFZSUrPcDivYlSrfB1zfGhlz2-zA"

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

with driver.session() as session:    
    print(session.run("RETURN 1 AS ok").single()["ok"])

1


In [62]:
from neo4j import GraphDatabase

# 본인의 Neo4j 연결 정보로 교체 (Neo4j Aura 무료 인스턴스 권장)
NEO4J_URI = "neo4j+s://ce6843cb.databases.neo4j.io"#실제 uri로 변경
NEO4J_USER = "ce6843cb" #실제 user로 변경
NEO4J_PASSWORD = "cNULIOfPxWEBZ1IXFZSUrPcDivYlSrfB1zfGhlz2-zA"  #이부분을 실제 패스워드로 변경

driver1 = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_query(query, parameters=None):
    with driver1.session() as session:
        result = session.run(query, parameters or {})
        return [record.data() for record in result]

# 연결 테스트
try:
    test = run_query("RETURN 'Neo4j 연결 성공' AS message")
    print(test)
except Exception as e:
    print("연결 실패:", e)

[{'message': 'Neo4j 연결 성공'}]


In [63]:
# 기존 데이터 초기화 (선택)
run_query("MATCH (n) DETACH DELETE n")
print("기존 그래프 초기화 완료")


기존 그래프 초기화 완료


In [64]:
# Cypher에서 사용할 파라미터 형태로 변환 (4M_분류 -> 4M분류는 속성명 사용 불가하므로 별도 키로 매핑)
reports_for_cypher = [
    {
        "설비명": r["설비명"],
        "불량유형": r["불량유형"],
        "M4분류": r["4M_분류"],
        "담당파트": r["담당파트"],
        "report_id": r["report_id"],
    }
    for r in reports
]


# reports.json 전체를 Cypher로 적재
create_query = '''
UNWIND $reports AS report
MERGE (eq:설비 {name: report.설비명})
MERGE (defect:불량유형 {name: report.불량유형})
MERGE (m4:M4분류 {name: report.M4분류})
MERGE (dept:담당파트 {name: report.담당파트})
MERGE (eq)-[:발생 {report_id: report.report_id}]->(defect)
MERGE (defect)-[:원인분류 {report_id: report.report_id}]->(m4)
MERGE (m4)-[:조치담당 {report_id: report.report_id}]->(dept)
'''


run_query(create_query, {"reports": reports_for_cypher})
print(f"{len(reports_for_cypher)}건 적재 완료")


150건 적재 완료


## 4. Cypher 쿼리 기본: 기본 조회 및 Multi-hop 탐색


In [65]:
# 기본 패턴 매칭: 특정 설비가 일으킨 불량유형 조회
query1 = '''
MATCH (eq:설비)-[:발생]->(defect:불량유형)
WHERE eq.name CONTAINS 'CNC'
RETURN eq.name AS 설비, defect.name AS 불량유형
LIMIT 10
'''
result1 = run_query(query1)
for row in result1:
    print(row)


{'설비': 'CNC 머신 A-1호기', '불량유형': '치수불량'}
{'설비': 'CNC 머신 A-2호기', '불량유형': '표면결함-스크래치/크랙'}
{'설비': 'CNC 머신 B-1호기', '불량유형': '조립불량'}
{'설비': 'CNC 머신 A-3호기', '불량유형': '치수불량'}
{'설비': 'CNC 머신 B-2호기', '불량유형': '표면결함-스크래치/크랙'}
{'설비': 'CNC 머신 C-1호기', '불량유형': '치수불량'}
{'설비': 'CNC 머신 C-2호기', '불량유형': '조립불량'}
{'설비': 'CNC 머신 D-1호기', '불량유형': '치수불량'}
{'설비': 'CNC 머신 D-2호기', '불량유형': '표면결함-스크래치/크랙'}
{'설비': 'CNC 머신 E-1호기', '불량유형': '조립불량'}


In [66]:
# Multi-hop 경로 탐색: 설비 -> 불량유형 -> 4M분류 -> 담당파트 (3-hop)
query2 = '''
MATCH path = (eq:설비)-[:발생]->(:불량유형)-[:원인분류]->(:M4분류)-[:조치담당]->(dept:담당파트)
WHERE eq.name CONTAINS '실린더블록'
RETURN eq.name AS 설비, dept.name AS 담당파트, length(path) AS 경로길이
LIMIT 10
'''
result2 = run_query(query2)
for row in result2:
    print(row)


In [67]:
# 특정 4M 분류와 관련된 모든 불량 사례 집계
query3 = '''
MATCH (defect:불량유형)-[:원인분류]->(m4:M4분류 {name: 'Machine'})
MATCH (eq:설비)-[:발생]->(defect)
RETURN defect.name AS 불량유형, count(DISTINCT eq) AS 관련설비수
ORDER BY 관련설비수 DESC
'''
result3 = run_query(query3)
for row in result3:
    print(row)


{'불량유형': '치수불량', '관련설비수': 23}
{'불량유형': '표면결함-스크래치/크랙', '관련설비수': 23}
{'불량유형': '이물질혼입', '관련설비수': 23}
{'불량유형': '조립불량', '관련설비수': 21}
{'불량유형': '누설', '관련설비수': 14}


- Cypher의 `(노드)-[관계]->(노드)` 패턴이 화살표 방향 그대로 관계를 표현해 직관적
- `MATCH path = (...)−[*1..3]->(...)` 형태로 가변 길이 경로 탐색도 가능
- NetworkX(인메모리, 소규모 프로토타입)와 Neo4j(영속 저장, 대규모 질의 최적화)의 차이


## 5. LLM 기반 엔티티·관계 추출 (Ollama 활용)

- 위에서는 구조화된 필드(설비명, 불량유형 등)를 그대로 그래프 노드로 사용
- 아래는 report_text에서 직접 LLM으로 엔티티·관계를 추출
- GraphRAG 인덱싱과 동일한 원리


In [80]:
!pkill -9 ollama

In [81]:
import subprocess
import time

# 백그라운드로 Ollama 서버 실행
process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(5)  # 서버 기동 대기
print("Ollama 서버 시작됨 (PID:", process.pid, ")")


Ollama 서버 시작됨 (PID: 19142 )


In [82]:
!ollama pull llama3:8b-instruct-q4_0

In [83]:
!ollama list

NAME                              ID              SIZE      MODIFIED               
llama3:8b-instruct-q4_0           365c0bd3c000    4.7 GB    Less than a second ago    
engine-diagnosis-expert:latest    d41b1ccf5af8    2.0 GB    About an hour ago         
qwen2.5:3b                        357c53fb659c    1.9 GB    About an hour ago         
llama3.2:3b                       a80c4f17acd5    2.0 GB    About an hour ago         
gemma:2b                          b50d6c999e59    1.7 GB    About an hour ago         
mistral:7b-instruct-q2_K          1344ecf13c2e    3.1 GB    About an hour ago         


In [99]:
import requests

OLLAMA_URL = "http://localhost:11434/api/generate"

def ask_ollama(model, prompt, temperature=0.1, timeout=900):
    response = requests.post(
        OLLAMA_URL,
        json={"model": model, "prompt": prompt, "stream": False,
              "options": {"temperature": temperature, "num_predict": 200}},
        timeout=timeout,
    )
    return response.json().get("response", "")

In [102]:
extraction_prompt_template = '''
당신은 정보 추출 엔진입니다. 아래 [분석할 보고서]를 읽고, 그 보고서 안의 실제 내용으로만 triple을 최대 3개 만드세요.
아래 [예시]는 형식만 보여주는 것이며, 예시의 내용을 답으로 쓰면 안 됩니다.

[예시 - 형식만 참고, 이 내용을 쓰지 마세요]
[
  {{"head": "용접기 #1호기", "relation": "발생", "tail": "누설"}},
  {{"head": "누설", "relation": "원인", "tail": "가스압력 이상"}}
]

[분석할 보고서 - 이 내용으로 triple을 만드세요]
{report_text}

[지금 위 보고서를 분석한 결과 JSON 배열]

'''

In [100]:
#reports = ask_ollama("llama3:8b-instruct-q4_0", extraction_prompt_template)
import json
with open("reports.json", "r", encoding="utf-8") as f:
    reports = json.load(f)

In [103]:
sample_report = reports[0]["report_text"]
prompt = extraction_prompt_template.format(report_text=sample_report)

print("입력 보고서:", sample_report)
print()
extraction_result = ask_ollama("qwen2.5:3b", prompt)  #llama3:8b-instruct-q4_0
print("=== 추출 결과 ===")
print(extraction_result)


입력 보고서: 2024년 1월 3일 09:15, CNC 머신 A-1호기에서 가공 완료된 알루미늄 브래킷의 기준 홀 직경이 도면 12.00mm 대비 12.18mm로 측정됨을 확인함. 동일 로트 20개 중 7개에서 +0.16~+0.18mm 오차가 반복되었고 스핀들 진동은 0.18mm 수준으로 확인됨. 설비보전팀 검토 결과 스핀들 베어링 마모에 따른 런아웃 발생으로 판단됨. 베어링 점검 및 교체 후 런아웃을 재측정하고 시험 가공품 5개의 홀 직경이 12.00±0.02mm 범위인지 검증함. 심각도는 상으로 분류하며, 향후 스핀들 진동 0.12mm 초과 시 즉시 설비보전팀 점검 요청 기준을 등록함.

=== 추출 결과 ===
[
  {"head": "CNC 머신 A-1호기", "relation": "발생", "tail": "알루미늄 브래킷의 기준 홀 직경"},
  {"head": "알루미늄 브래킷의 기준 홀 직경", "relation": "오차", "tail": "+0.16~+0.18mm"},
  {"head": "CNC 머신 A-1호기", "relation": "발생", "tail": "스핀들 진동"}
]


In [104]:
import json as json_lib
import re

def parse_triples(raw_text):
    # LLM 응답에서 JSON 배열 부분만 추출
    match = re.search(r"\[.*\]", raw_text, re.DOTALL)
    if not match:
        return []
    try:
        return json_lib.loads(match.group())
    except json_lib.JSONDecodeError:
        return []

triples = parse_triples(extraction_result)
print(f"추출된 triple 수: {len(triples)}")
for t in triples:
    print(t)


추출된 triple 수: 3
{'head': 'CNC 머신 A-1호기', 'relation': '발생', 'tail': '알루미늄 브래킷의 기준 홀 직경'}
{'head': '알루미늄 브래킷의 기준 홀 직경', 'relation': '오차', 'tail': '+0.16~+0.18mm'}
{'head': 'CNC 머신 A-1호기', 'relation': '발생', 'tail': '스핀들 진동'}


In [106]:
# 여러 보고서에 대해 일괄 추출 (5건 샘플)
all_triples = []
for r in reports[:5]:
    prompt = extraction_prompt_template.format(report_text=r["report_text"])
    raw = ask_ollama("llama3:8b-instruct-q4_0", prompt)
    triples = parse_triples(raw)
    for t in triples:
        t["report_id"] = r["report_id"]
    all_triples.extend(triples)
    print(f"{r['report_id']}: {len(triples)}개 triple 추출")

print(f"\n전체 추출 triple 수: {len(all_triples)}")


RPT-001: 3개 triple 추출
RPT-002: 3개 triple 추출
RPT-003: 3개 triple 추출
RPT-004: 3개 triple 추출
RPT-005: 3개 triple 추출

전체 추출 triple 수: 15


- 구조화 필드 기반 그래프보다 훨씬 세밀한 엔티티(부품명, 구체적 증상)가 추출
- 추출 품질이 LLM 성능에 의존하며, 동일 엔티티가 다른 표현으로 추출되는 entity resolution 문제가 발생할 수 있음
- 이 entity resolution 이슈는 GraphRAG의 커뮤니티 탐지 단계에서 일부 완화됨


In [107]:
all_triples

[{'head': '스핀들 베어링 마모',
  'relation': '원인',
  'tail': '런아웃',
  'report_id': 'RPT-001'},
 {'head': '런아웃', 'relation': '발생', 'tail': '홀 직경 오차', 'report_id': 'RPT-001'},
 {'head': '홀 직경 오차',
  'relation': '원인',
  'tail': '스핀들 진동',
  'report_id': 'RPT-001'},
 {'head': 'CNC 머신 A-2호기',
  'relation': '발생',
  'tail': '미세 스크래치',
  'report_id': 'RPT-002'},
 {'head': '마모된 엔드밀 절삭날',
  'relation': '원인',
  'tail': '스클래치',
  'report_id': 'RPT-002'},
 {'head': '절삭온도', 'relation': '상승', 'tail': '165°C', 'report_id': 'RPT-002'},
 {'head': '체결 볼트 #1', 'relation': '발생', 'tail': '누설', 'report_id': 'RPT-003'},
 {'head': '누설',
  'relation': '원인',
  'tail': '토크렌치 설정값 유지',
  'report_id': 'RPT-003'},
 {'head': '작업지시서',
  'relation': '추가',
  'tail': '품목별 체결 토크 확인 체크 항목',
  'report_id': 'RPT-003'},
 {'head': '사출성형기 #1호기',
  'relation': '발생',
  'tail': '크랙',
  'report_id': 'RPT-004'},
 {'head': '크랙',
  'relation': '원인',
  'tail': '금형온도 조절기 히터 성능 저하',
  'report_id': 'RPT-004'},
 {'head': '금형온도 조절기 히터 성능 저하',
  'rel

In [ ]:
#pyvis 시각화

In [108]:
# pyvis 시각화
from pyvis.network import Network

net = Network(notebook=True, cdn_resources="in_line",
              height="600px", width="100%", directed=True)

# report_id별 색상
report_colors = {
    "RPT-001": "#4C72B0",
    "RPT-002": "#DD8452",
    "RPT-003": "#55A868",
    "RPT-004": "#C44E52",
    "RPT-005": "#8172B2",
}

added_nodes = set()

for t in all_triples:
    color = report_colors.get(t["report_id"], "#999999")

    if t["head"] not in added_nodes:
        net.add_node(t["head"], label=t["head"], color=color)
        added_nodes.add(t["head"])

    if t["tail"] not in added_nodes:
        net.add_node(t["tail"], label=t["tail"], color=color)
        added_nodes.add(t["tail"])

    net.add_edge(t["head"], t["tail"],
                 label=t["relation"],
                 title=t["report_id"])

net.set_options("""
{
  "edges": {
    "arrows": { "to": { "enabled": true } },
    "font": { "size": 11 }
  },
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -50,
      "springLength": 120
    },
    "solver": "forceAtlas2Based"
  }
}
""")

output_path = "all_triples_graph.html"
net.show(output_path)
print(f"시각화 저장: {output_path}")

all_triples_graph.html
시각화 저장: all_triples_graph.html


In [110]:
#neo4j 적재

In [109]:
# Neo4j 적재
from neo4j import GraphDatabase

NEO4J_URI = "neo4j+s://ce6843cb.databases.neo4j.io"#실제 uri로 변경
NEO4J_USER = "ce6843cb" #실제 user로 변경
NEO4J_PASSWORD = "cNULIOfPxWEBZ1IXFZSUrPcDivYlSrfB1zfGhlz2-zA"  #이부분을 실제 패스워드로 변경

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def load_triples(tx, triples):
    for t in triples:
        tx.run("""
            MERGE (h:Entity {name: $head})
            MERGE (t:Entity {name: $tail})
            MERGE (h)-[r:RELATION {type: $relation, report_id: $report_id}]->(t)
        """,
        head=t["head"],
        tail=t["tail"],
        relation=t["relation"],
        report_id=t["report_id"])

with driver.session() as session:
    session.execute_write(load_triples, all_triples)

print(f"Neo4j 적재 완료: {len(all_triples)}개 triple")
driver.close()

Neo4j 적재 완료: 15개 triple


In [112]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
with driver.session() as session:
    result = session.run("""
        MATCH (h)-[r:RELATION]->(t)
        RETURN h.name AS head, r.type AS relation, t.name AS tail, r.report_id AS report_id
        ORDER BY r.report_id
    """)
    
    records = result.data()

driver.close()

# 출력
for rec in records:
    print(f"[{rec['report_id']}] {rec['head']} --{rec['relation']}--> {rec['tail']}")

print(f"\n총 {len(records)}개 레코드")

[RPT-001] 스핀들 베어링 마모 --원인--> 런아웃
[RPT-001] 런아웃 --발생--> 홀 직경 오차
[RPT-001] 홀 직경 오차 --원인--> 스핀들 진동
[RPT-002] CNC 머신 A-2호기 --발생--> 미세 스크래치
[RPT-002] 마모된 엔드밀 절삭날 --원인--> 스클래치
[RPT-002] 절삭온도 --상승--> 165°C
[RPT-003] 체결 볼트 #1 --발생--> 누설
[RPT-003] 누설 --원인--> 토크렌치 설정값 유지
[RPT-003] 작업지시서 --추가--> 품목별 체결 토크 확인 체크 항목
[RPT-004] 사출성형기 #1호기 --발생--> 크랙
[RPT-004] 크랙 --원인--> 금형온도 조절기 히터 성능 저하
[RPT-004] 금형온도 조절기 히터 성능 저하 --원인--> 잔류응력 증가
[RPT-005] CNC 머신 A-3호기 --발생--> 기어 하우징 포켓 깊이 얕게 가공
[RPT-005] Z축 원점 보정값 --원인--> 불량 수량
[RPT-005] 작업자 --원인--> Z축 오프셋 보정 누락

총 15개 레코드


In [113]:
import pandas as pd
df = pd.DataFrame(records)
df

,head,relation,tail,report_id
0,스핀들 베어링 마모,원인,런아웃,RPT-001
1,런아웃,발생,홀 직경 오차,RPT-001
2,홀 직경 오차,원인,스핀들 진동,RPT-001
3,CNC 머신 A-2호기,발생,미세 스크래치,RPT-002
4,마모된 엔드밀 절삭날,원인,스클래치,RPT-002
5,절삭온도,상승,165°C,RPT-002
6,체결 볼트 #1,발생,누설,RPT-003
7,누설,원인,토크렌치 설정값 유지,RPT-003
8,작업지시서,추가,품목별 체결 토크 확인 체크 항목,RPT-003
9,사출성형기 #1호기,발생,크랙,RPT-004
